**Expenditure** **Analyser** -- **project** **2**

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 60)
df = pd.read_csv("rahul_transactions.csv")

date_formats = ['%Y-%m-%d', '%d/%m/%y', '%d-%b-%y', '%d %b %Y']
parsed_date = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
for fmt in date_formats:
    still_missing = parsed_date.isna()
    attempt = pd.to_datetime(df.loc[still_missing, 'Date'], format=fmt, errors='coerce')
    parsed_date.loc[still_missing] = attempt
df['date'] = parsed_date

# --- Amounts: strip ₹ / Rs. / commas -> numeric ---
df['amount'] = (
    df['Amount']
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

# --- Type: DR/Debit, CR/Credit -> canonical lowercase ---
type_map = {'DR': 'debit', 'Debit': 'debit', 'CR': 'credit', 'Credit': 'credit'}
df['type_clean'] = df['Type'].map(type_map)

# --- Mode: blank string -> NaN ---
df['Mode'] = df['Mode'].replace('', np.nan)

# --- Drop exact duplicate rows ---
before = len(df)
df = df.drop_duplicates(subset=['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref'])
dupes_dropped = before - len(df)

# --- Drop rows that still failed to parse ---
bad_dates = df['date'].isna().sum()
bad_amounts = df['amount'].isna().sum()
df = df.dropna(subset=['date', 'amount']).reset_index(drop=True)

# --- Time-derived columns (Section 5.1 of the brief) ---
df['hour'] = df['Time'].str[:2].astype(int)
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['day_of_week'] = df['date'].dt.day_name()

print(f"Parsed {len(df)} transactions across 6 months. "
      f"Dropped {dupes_dropped} duplicates. "
      f"{bad_amounts} unparseable amounts, {bad_dates} unparseable dates.")
df.dtypes[['date', 'amount', 'type_clean', 'hour', 'month']]

VENDOR_MAP = {
    # --- Food Delivery ---
    'Swiggy':          ['SWIGGY', 'BUNDL'],
    'Zomato':          ['ZOMATO'],
    # --- Quick Commerce (checked before generic Swiggy/Zomato rules) ---
    'Swiggy Instamart': ['INSTAMART', 'SWIGGY-INSTAMART', 'SWIGGY INSTAMART'],
    'Blinkit':         ['BLINKIT', 'GROFERS'],
    'Zepto':           ['ZEPTO', 'KIRANAKART'],
    'BigBasket':       ['BIGBASKET', 'INNOVATIVE RETAIL'],
    'DMart':           ['DMART', 'AVENUE SUPERMARTS'],
    # --- E-commerce ---
    'Amazon':          ['AMAZON', 'AMZN', 'FSN E-COMMERCE'],
    'Flipkart':        ['FLIPKART', 'FKART'],
    'Myntra':          ['MYNTRA'],
    'Nykaa':           ['NYKAA'],
    # --- Transport ---
    'Uber':            ['UBER', 'ROPPEN'],
    'Ola':             ['OLA', 'ANI TECHNOLOGIES'],
    'Rapido':          ['RAPIDO'],
    'BMTC':            ['BMTC', 'TUMMOC'],
    # --- Cafe ---
    'Starbucks':       ['STARBUCKS', 'TATA STARBUCKS'],
    'Third Wave Coffee': ['THIRDWAVE', 'THIRD WAVE'],
    'Cafe Coffee Day': ['CCD', 'COFFEE DAY'],
    # --- Restaurants ---
    'Dineout':         ['DINEOUT'],
    'Restaurant (general)': ['RESTAURANT', 'MEGHANA FOODS', 'TRUFFLES', 'EMPIRE RESTAURANT'],
    # --- Subscriptions ---
    'Netflix':         ['NETFLIX'],
    'Spotify':         ['SPOTIFY'],
    'Amazon Prime':    ['PRIME VIDEO', 'AMZN PRIME', 'AMAZON-PRIME', 'AMAZON PRIME'],
    'Hotstar':         ['HOTSTAR', 'STAR INDIA', 'TWC INDIA'],
    'JioFiber':        ['JIOFIBER'],
    # --- Utilities ---
    'Airtel':          ['AIRTEL', 'BHARTI AIRTEL'],
    'Vodafone Idea':   ['VODAFONE', 'VI POSTPAID', 'UPI-VI-RECHARGE'],
    'Jio Recharge':    ['JIORECHARGE', 'RELIANCE JIO'],
    'BESCOM':          ['BESCOM', 'BANGALORE ELEC'],
    'BWSSB':           ['BWSSB'],
    # --- Investments ---
    'Zerodha':         ['ZERODHA'],
    'Groww':           ['GROWW'],
    # --- Fuel ---
    'BPCL':            ['BPCL'],
    'IOC':             ['IOC', 'INDIAN OIL'],
    'HP Petrol':       ['HP PETROL'],
    # --- Entertainment ---
    'BookMyShow':      ['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE'],
    # --- Fixed / non-merchant ---
    'Rent':            ['RENT-LANDLORD'],
    'Salary':          ['SALARY'],
}

def extract_vendor(desc):
    desc_upper = desc.upper()

    if desc.startswith('ATM-WDL'):
        return 'Cash Withdrawal'

    for vendor, keywords in VENDOR_MAP.items():
        for kw in keywords:
            if kw in desc_upper:
                return vendor

    # Friend-to-friend transfers: UPI-<FirstName>-<digits>@handle
    if desc.startswith('UPI-') and '@' in desc:
        return 'P2P Transfer'

    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

print('Unique canonical vendors:', df['vendor_clean'].nunique())
print(df['vendor_clean'].value_counts().head(10))
uncategorised = df[df['vendor_clean'] == 'Uncategorised']['Description'].unique()
print(f"\nUncategorised ({len(uncategorised)}):", list(uncategorised))

CATEGORY_MAP = {
    'Food Delivery':   ['Swiggy', 'Zomato'],
    'Quick Commerce':  ['Swiggy Instamart', 'Blinkit', 'Zepto', 'BigBasket', 'DMart'],
    'Ecommerce':       ['Amazon', 'Flipkart', 'Myntra', 'Nykaa'],
    'Transport':       ['Uber', 'Ola', 'Rapido', 'BMTC'],
    'Cafe':            ['Starbucks', 'Third Wave Coffee', 'Cafe Coffee Day'],
    'Restaurants':     ['Dineout', 'Restaurant (general)'],
    'Subscriptions':   ['Netflix', 'Spotify', 'Amazon Prime', 'Hotstar', 'JioFiber'],
    'Utilities':       ['Airtel', 'Vodafone Idea', 'Jio Recharge', 'BESCOM', 'BWSSB'],
    'Investments':     ['Zerodha', 'Groww'],
    'Fuel':            ['BPCL', 'IOC', 'HP Petrol'],
    'Entertainment':   ['BookMyShow'],
    'Personal Transfer': ['P2P Transfer'],
    'Cash Withdrawal':   ['Cash Withdrawal'],
    'Income':            ['Salary'],
    'Fixed - Rent':      ['Rent'],
}

vendor_to_category = {v: cat for cat, vendors in CATEGORY_MAP.items() for v in vendors}
df['category'] = df['vendor_clean'].map(vendor_to_category)

print(df['category'].value_counts())
print('\nUnmapped vendors left:', df[df['category'].isna()]['vendor_clean'].unique())

total_credits = df.loc[df['type_clean'] == 'credit', 'amount'].sum()
total_debits  = df.loc[df['type_clean'] == 'debit', 'amount'].sum()
net_change    = total_credits - total_debits
savings_rate  = net_change / total_credits * 100

debit_df  = df[df['type_clean'] == 'debit']
spend_df  = debit_df[~debit_df['category'].isin(['Personal Transfer', 'Cash Withdrawal'])]
total_spend = spend_df['amount'].sum()

top_categories = spend_df.groupby('category')['amount'].sum().sort_values(ascending=False)
top_vendors    = spend_df.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False)

print(f"Total credits : Rs. {total_credits:,.0f}")
print(f"Total debits  : Rs. {total_debits:,.0f}")
print(f"Net change    : Rs. {net_change:,.0f}")
print(f"Savings rate  : {savings_rate:.1f}%")
print()
print('TOP 5 CATEGORIES (% of spend, excl. transfers/withdrawals)')
for cat, amt in top_categories.head(5).items():
    print(f"  {cat:<18} {amt/total_spend*100:5.1f}%   Rs. {amt:,.0f}")
print()
print('TOP 5 VENDORS BY SPEND')
for v, amt in top_vendors.head(5).items():
    n = (spend_df['vendor_clean'] == v).sum()
    print(f"  {v:<18} Rs. {amt:>10,.0f}  ({n} orders)")
month_pivot = spend_df.pivot_table(values='amount', index='category', columns='month_name',
                                    aggfunc='sum', fill_value=0)
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
month_pivot = month_pivot[[m for m in month_order if m in month_pivot.columns]]
print(month_pivot.round(0))

first_col, last_col = month_pivot.columns[0], month_pivot.columns[-1]
pct_change = ((month_pivot[last_col] - month_pivot[first_col])
              / month_pivot[first_col].replace(0, np.nan)) * 100
pct_change = pct_change.sort_values(ascending=False)
print(f"\nBiggest growth ({first_col}->{last_col}): {pct_change.index[0]} ({pct_change.iloc[0]:+.1f}%)")
print(f"Biggest decline ({first_col}->{last_col}): {pct_change.index[-1]} ({pct_change.iloc[-1]:+.1f}%)")

categories = sorted(spend_df['category'].unique())
cat_index = {c: i for i, c in enumerate(categories)}
matrix = np.zeros((len(categories), 24), dtype=int)
for _, row in spend_df.iterrows():
    matrix[cat_index[row['category']], row['hour']] += 1

food = spend_df[spend_df['category'] == 'Food Delivery']
late_night_pct = ((food['hour'] >= 21) | (food['hour'] < 2)).mean() * 100
print(f"Food Delivery orders 21:00-02:00: {late_night_pct:.1f}%")

cafe = spend_df[spend_df['category'] == 'Cafe']
morning_pct = ((cafe['hour'] >= 8) & (cafe['hour'] <= 11)).mean() * 100
print(f"Cafe orders 8-11 AM: {morning_pct:.1f}%")

spend_df = spend_df.copy()
category_mean = spend_df.groupby('category')['amount'].transform('mean')
category_std  = spend_df.groupby('category')['amount'].transform('std')
spend_df['z_score'] = (spend_df['amount'] - category_mean) / category_std

anomalies = spend_df[spend_df['z_score'] > 2].sort_values('z_score', ascending=False)
print(f"Anomalies flagged (z > 2): {len(anomalies)}")
print()
for _, r in anomalies.head(5).iterrows():
    print(f"  {r['date'].strftime('%d %b')} - {r['vendor_clean']:<12} "
          f"Rs. {r['amount']:>8,.0f}  (z={r['z_score']:.1f}, category={r['category']})")

total_credits = df.loc[df['type_clean'] == 'credit', 'amount'].sum()
total_debits  = df.loc[df['type_clean'] == 'debit', 'amount'].sum()
cat_pct = spend_df.groupby('category')['amount'].sum() / spend_df['amount'].sum() * 100

def is_foodie(cat_pct):
    pct = cat_pct.reindex(['Food Delivery', 'Restaurants', 'Cafe']).fillna(0).sum()
    return pct > 25, f"{pct:.1f}% on Food"

def is_quick_commerce_junkie(cat_pct):
    pct = cat_pct.get('Quick Commerce', 0)
    return pct > 15, f"{pct:.1f}% on Q-commerce"

def is_shopaholic(cat_pct):
    pct = cat_pct.get('Ecommerce', 0)
    return pct > 15, f"{pct:.1f}% on E-commerce"

def is_investor(cat_pct):
    pct = cat_pct.get('Investments', 0)
    return pct > 15, f"{pct:.1f}% on Investments"

def is_late_night_snacker(spend_df):
    food = spend_df[spend_df['category'] == 'Food Delivery']
    if len(food) == 0:
        return False, "0.0% food orders after 9 PM"
    late = ((food['hour'] >= 21) | (food['hour'] < 2)).mean() * 100
    return late > 50, f"{late:.1f}% food orders 9 PM-2 AM"

def is_cab_commuter(cat_pct):
    pct = cat_pct.get('Transport', 0)
    return pct > 10, f"{pct:.1f}% on Transport"

def is_subscription_lover(spend_df):
    n = spend_df[spend_df['category'] == 'Subscriptions']['vendor_clean'].nunique()
    return n >= 5, f"{n} distinct subscription vendors"

def is_yolo_spender(total_credits, total_debits):
    rate = (total_credits - total_debits) / total_credits * 100
    return rate < 10, f"savings rate {rate:.1f}%"

def is_disciplined_saver(total_credits, total_debits):
    rate = (total_credits - total_debits) / total_credits * 100
    return rate > 40, f"savings rate {rate:.1f}%"

rules = [
    ('THE FOODIE', is_foodie(cat_pct)),
    ('THE QUICK COMMERCE JUNKIE', is_quick_commerce_junkie(cat_pct)),
    ('THE SHOPAHOLIC', is_shopaholic(cat_pct)),
    ('THE INVESTOR', is_investor(cat_pct)),
    ('THE LATE-NIGHT SNACKER', is_late_night_snacker(spend_df)),
    ('THE CAB COMMUTER', is_cab_commuter(cat_pct)),
    ('THE SUBSCRIPTION LOVER', is_subscription_lover(spend_df)),
    ('THE YOLO SPENDER', is_yolo_spender(total_credits, total_debits)),
    ('THE DISCIPLINED SAVER', is_disciplined_saver(total_credits, total_debits)),
]

print("RAHUL'S SPENDING ARCHETYPES")
for name, (matched, metric) in rules:
    flag = '-> MATCHED ' if matched else '   no match'
    print(f"  {flag} {name:<28} ({metric})")

BAR_UNIT = 2
top_cat = spend_df.groupby('category')['amount'].sum().sort_values(ascending=False)
top_vendor = spend_df.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False)
net = total_credits - total_debits
savings_rate = net / total_credits * 100
total_spend = spend_df['amount'].sum()

print('=' * 64)
print(' SpendDNA REPORT - RAHUL SHARMA')
print(f' 6 months - {len(df)} transactions - Jan to Jun 2024')
print('=' * 64)
print(' EXECUTIVE SUMMARY')
print(f'   Total credits : Rs. {total_credits:>12,.0f}')
print(f'   Total debits  : Rs. {total_debits:>12,.0f}')
print(f"   Net change    : Rs. {net:>12,.0f} ({'overspending' if net < 0 else 'saving'})")
print(f'   Savings rate  : {savings_rate:.1f}%')
print(f'   Transactions  : {len(df)}')
print(f"   Unique vendors: {df['vendor_clean'].nunique()}")
print()
print(' TOP CATEGORIES (% of spend)')
for cat, amt in top_cat.head(5).items():
    pct = amt / total_spend * 100
    print(f"   {cat:<16} {'#' * int(pct / BAR_UNIT):<12} {pct:5.1f}%  Rs. {amt:>10,.0f}")
print()
print(' TOP VENDORS')
for v, amt in top_vendor.head(5).items():
    n = (spend_df['vendor_clean'] == v).sum()
    print(f'   {v:<16} Rs. {amt:>10,.0f}  ({n} orders)')
print()
print(' TOP ANOMALIES (z > 2, within category)')
for _, r in anomalies.head(5).iterrows():
    print(f"   {r['date'].strftime('%d %b')} - {r['vendor_clean']:<12} Rs. {r['amount']:>8,.0f} (z={r['z_score']:.1f})")
print('=' * 64)


Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 0 unparseable dates.
Unique canonical vendors: 39
vendor_clean
Swiggy                  223
Zomato                  121
Amazon                   89
Ola                      87
Uber                     85
Zepto                    71
Restaurant (general)     63
Blinkit                  55
Flipkart                 47
Starbucks                42
Name: count, dtype: int64

Uncategorised (0): []
category
Food Delivery        344
Transport            250
Quick Commerce       187
Ecommerce            172
Cafe                  90
Restaurants           73
Utilities             42
Subscriptions         41
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Fixed - Rent           6
Name: count, dtype: int64

Unmapped vendors left: []
Total credits : Rs. 509,774
Total debits  : Rs. 1,678,901
Net change    : Rs. -1,16